# 05 - RQ1: So sánh 4 nhóm đặc trưng thủ công trên Random Forest cố định (S3-05 & S3-06)

**Bài tập lớn môn Học máy:** Phân loại 4 nhóm ảnh X-quang ngực (`COVID`, `Lung_Opacity`, `Normal`, `Viral Pneumonia`).  
**Câu hỏi nghiên cứu 1 (RQ1):** Nhóm đặc trưng thủ công nào (`hist`: Histogram độ xám 64 chiều; `glcmlbp`: GLCM + LBP 88 chiều; `hog`: HOG 10.404 chiều; hoặc `ghep`: Histogram + GLCM + LBP 152 chiều) đạt Macro-F1 validation cao nhất với bộ phân loại cố định `RandomForestClassifier(n_estimators=500, class_weight="balanced_subsample")` trên 3 seed (`42, 43, 44`)?  
**Người phụ trách (Code Owner):** Lê Hải Lý (S3-05) & Trần Nguyễn Đức (S3-06).

## Nội dung thực hiện theo `results/threshold_v3.md`:
1. **Kiểm toán 12 run (`results/rq1_<features>_rf_s<seed>`):** Xác minh các cổng hợp lệ H1–H7 và đối chiếu mốc nền `B0 = 0,1627`, `A0 = 0,2397`.
2. **Thống kê ngân sách tính toán (H5):** Đối chiếu thời gian trích đặc trưng từ `results/summary_full.json` và thời gian huấn luyện từ `metrics.json`.
3. **Xuất và trình bày bộ báo cáo bắt buộc R1, R2, R4, R5 (`src/bootstrap.py`):**
   - **R1:** Macro-F1 và Precision/Recall/F1 từng lớp (trung bình ± độ lệch chuẩn mẫu qua 3 seed) kèm khoảng tin cậy 95% bootstrap (2.000 lần).
   - **Mục 5 (Phép so sánh chính):** Kiểm định bootstrap ghép cặp 2.000 lần so với cấu hình tham chiếu `hist`.
   - **R2:** Ma trận nhầm lẫn 4×4 và ma trận con 2×2 giữa `Normal` và `Lung_Opacity`.
   - **R4:** Sai số chuẩn lấy mẫu xấp xỉ trên toàn tập validation ($n = 3.174$) và lớp `Viral Pneumonia` ($n = 203$).
   - **R5:** Các giới hạn phương pháp luận.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

# Định vị thư mục gốc của repository
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "results" / "threshold_v3.md").exists() and (REPO_ROOT.parent / "results" / "threshold_v3.md").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.bootstrap import build_report, macro_f1_from_labels
from src.metrics import DEFAULT_CLASS_NAMES, compute_metrics

RESULTS_DIR = REPO_ROOT / "results"
REPORT_DIR = RESULTS_DIR / "report_rq1"
FEATURES_ORDER = ["hist", "glcmlbp", "hog", "ghep"]
SEEDS_ORDER = [42, 43, 44]
B0_MACRO_F1 = 0.1627
A0_BEST_MACRO_F1 = 0.2397

print(f"Thư mục gốc dự án: {REPO_ROOT}")
print(f"NumPy: {np.__version__} | Pandas: {pd.__version__}")


## 1. Kiểm toán 12 Run thực nghiệm và Cổng hợp lệ H1–H6

Mỗi run trong `results/rq1_<features>_rf_s<seed>` được kiểm tra:
- Tồn tại đủ 4 tệp `config.yaml`, `metrics.json`, `log`, `preds_val.csv` (Cổng H1).
- `allow_test = False`, `n_train = 14.815`, `n_val = 3.174`, mã băm `split_v1.csv` khớp chuẩn (Cổng H2, H6).
- `RandomForestClassifier` 500 cây, `class_weight = "balanced_subsample"`, `seed == random_state` (Cổng H3).
- Điểm `macro_f1` tính lại trực tiếp từ `preds_val.csv` khớp với `metrics.json` và vượt qua `B0 = 0,1627`, `A0 = 0,2397` (Cổng H4).


In [ ]:
EXPECTED_COMMIT = "4c0b9cf2db903e76d08235fd83be653366c67ffe"
EXPECTED_SPLIT_SHA256 = "f99e03d3d5cfc32f2bbc9befbdef45a07a256458a837a6d420a6658ed7583d98"

audit_rows = []
for feat in FEATURES_ORDER:
    for seed in SEEDS_ORDER:
        run_id = f"rq1_{feat}_rf_s{seed}"
        run_dir = RESULTS_DIR / run_id
        for req_file in ("config.yaml", "metrics.json", "log", "preds_val.csv"):
            assert (run_dir / req_file).is_file(), f"Thiếu {req_file} trong {run_id}"

        cfg = yaml.safe_load((run_dir / "config.yaml").read_text(encoding="utf-8"))
        met = json.loads((run_dir / "metrics.json").read_text(encoding="utf-8"))
        preds_df = pd.read_csv(run_dir / "preds_val.csv")

        recomp = compute_metrics(preds_df["y_true"].to_numpy(), preds_df["y_pred"].to_numpy())
        exact_f1 = macro_f1_from_labels(preds_df["y_true"].to_numpy(), preds_df["y_pred"].to_numpy())

        assert cfg["commit"] == EXPECTED_COMMIT
        assert cfg["split"]["sha256"] == EXPECTED_SPLIT_SHA256
        assert cfg["evaluation"]["allow_test"] is False
        assert cfg["hyperparameters"]["n_estimators"] == 500
        assert cfg["hyperparameters"]["class_weight"] == "balanced_subsample"
        assert cfg["seed"] == seed and cfg["hyperparameters"]["random_state"] == seed
        assert recomp["macro_f1"] == met["macro_f1"]
        assert recomp["accuracy"] == met["accuracy"]
        assert recomp["confusion_matrix"] == met["confusion_matrix"]
        assert exact_f1 > B0_MACRO_F1 and exact_f1 > A0_BEST_MACRO_F1

        sec = met["seconds"]
        audit_rows.append({
            "run_id": run_id,
            "features": feat,
            "seed": seed,
            "macro_f1_exact": exact_f1,
            "macro_f1_json": met["macro_f1"],
            "accuracy": met["accuracy"],
            "fit_cpu_s": sec["fit_cpu"],
            "fit_wall_s": sec["fit_wall"],
            "predict_wall_s": sec["predict_wall"],
        })

audit_df = pd.DataFrame(audit_rows)
print(f"Kiểm toán 12/12 run: ĐẠT TUYỆT ĐỐI (H1-H6).")
print(audit_df.to_string(index=False))


## 2. Ngân sách thời gian trích đặc trưng và huấn luyện (Cổng H5)

Tổng hợp thời gian CPU trích đặc trưng từ `results/summary_full.json` (đã đóng băng trong `results/rq1_cost_rank_record.md`) và thời gian huấn luyện trung bình của Random Forest trên 4 nhóm đặc trưng.


In [ ]:
summary_full = json.loads((RESULTS_DIR / "summary_full.json").read_text(encoding="utf-8"))
FEATURE_DIMS = {"hist": 64, "glcmlbp": 88, "hog": 10404, "ghep": 152}

cost_rows = []
for feat in FEATURES_ORDER:
    ext_train = summary_full["extract_cpu_seconds"]["train"][feat]
    ext_val = summary_full["extract_cpu_seconds"]["val"][feat]
    ext_per_img = summary_full["extract_cpu_seconds_per_image"][feat]
    sub_fit = audit_df[audit_df["features"] == feat]
    cost_rows.append({
        "features": feat,
        "n_dims": FEATURE_DIMS[feat],
        "extract_cpu_train_s": round(ext_train, 3),
        "extract_cpu_val_s": round(ext_val, 3),
        "extract_cpu_per_image_s": ext_per_img,
        "cost_rank": summary_full["cost_rank"].index(feat) + 1,
        "fit_cpu_mean_s": round(float(sub_fit["fit_cpu_s"].mean()), 2),
        "fit_wall_mean_s": round(float(sub_fit["fit_wall_s"].mean()), 2),
    })

cost_df = pd.DataFrame(cost_rows).sort_values("cost_rank")
print("=== BẢNG NGÂN SÁCH TÍNH TOÁN (THỨ HẠNG CHI PHÍ TỪ RẺ ĐẾN ĐẮT) ===")
print(cost_df.to_string(index=False))


## 3. Báo cáo bắt buộc R1 & Phép so sánh chính của RQ1 (`src/bootstrap.py`)

- Gọi `build_report` để tạo/cập nhật 8 tệp báo cáo chuẩn tại `results/report_rq1/`.
- Hiển thị bảng tổng hợp **R1** (Macro-F1 và Recall/Precision/F1 từng lớp qua 3 seed kèm 95% CI bootstrap 2.000 lần) và bảng kiểm định **Bootstrap ghép cặp so với tham chiếu `hist`**.


In [ ]:
report_paths = build_report(
    results_dir=RESULTS_DIR,
    out_dir=REPORT_DIR,
    features=FEATURES_ORDER,
    seeds=SEEDS_ORDER,
    reference="hist",
    n_boot=2000,
    rng_seed=42,
    expected_n=3174,
    threshold_path=RESULTS_DIR / "threshold_v3.md",
)

r1_summary_df = pd.read_csv(report_paths["r1_summary.csv"])
r1_boot_df = pd.read_csv(report_paths["r1_bootstrap_vs_reference.csv"])

display_r1_cols = [
    "features",
    "macro_f1_mean",
    "macro_f1_std",
    "ci95_low",
    "ci95_high",
    "recall_COVID_mean",
    "recall_Lung_Opacity_mean",
    "recall_Normal_mean",
    "recall_Viral Pneumonia_mean",
]
print("=== BẢNG R1: MACRO-F1 VÀ RECALL TỪNG LỚP (TRUNG BÌNH 3 SEED & 95% CI BOOTSTRAP) ===")
print(r1_summary_df[display_r1_cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\n=== KIỂM ĐỊNH BOOTSTRAP GHÉP CẶP 2.000 LẦN SO VỚI THAM CHIẾU HIST (MỤC 5 THRESHOLD_V3) ===")
print(r1_boot_df.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

# Xác định phép so sánh chính của RQ1: nhóm khác hist có macro-F1 trung bình cao nhất
non_ref_df = r1_summary_df[r1_summary_df["features"] != "hist"].sort_values("macro_f1_mean", ascending=False)
best_cand = str(non_ref_df.iloc[0]["features"])
primary_row = r1_boot_df[r1_boot_df["candidate"] == best_cand].iloc[0]
passes_rule = (float(primary_row["point_diff"]) >= 0.005) and (not bool(primary_row["contains_zero"]))
selected_feature_group = best_cand if passes_rule else "hist"

print(f"\nPhép so sánh chính RQ1: '{best_cand}' vs 'hist'")
print(f"  Hiệu số Macro-F1 (point_diff) = {primary_row['point_diff']:+.6f} (Ngưỡng >= +0.005)")
print(f"  95% CI Bootstrap ghép cặp     = [{primary_row['ci_low']:+.6f}, {primary_row['ci_high']:+.6f}] (contains_zero = {primary_row['contains_zero']})")
print(f"==> QUYẾT ĐỊNH CHỐT NHÓM ĐẶC TRƯNG ĐI TIẾP SANG RQ2 (S3-06): '{selected_feature_group}'")


## 4. Báo cáo bắt buộc R2 (Ma trận con `Normal – Lung_Opacity`), R4 (Sai số lấy mẫu) và R5 (Giới hạn)


In [ ]:
r2_df = pd.read_csv(report_paths["r2_normal_lung_opacity.csv"])
r4_df = pd.read_csv(report_paths["r4_sampling_error.csv"])
r5_str = report_paths["r5_gioi_han.md"].read_text(encoding="utf-8").strip()

print("=== BẢNG R2: MA TRẬN CON 2x2 NORMAL - LUNG OPACITY (TRUNG BÌNH 3 SEED) ===")
r2_mean = r2_df.groupby("features", as_index=False)[["LO_to_LO", "LO_to_Normal", "Normal_to_LO", "Normal_to_Normal"]].mean()
print(r2_mean.to_string(index=False, float_format=lambda x: f"{x:.1f}"))

print("\n=== BẢNG R4: SAI SỐ CHUẨN LẤY MẪU XẤP XỈ (TRUNG BÌNH 3 SEED) ===")
r4_mean = r4_df.groupby("features", as_index=False)[["p_accuracy", "se_all", "p_recall_viral", "se_viral"]].mean()
print(r4_mean.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\n=== R5: GIỚI HẠN NGHIÊN CỨU ===")
print(r5_str)
